## Male–Female Parallel Transport (Tansley 2022)

**Question:** Can the female trajectory predict male microglia dynamics?

**Strategy:** For each condition (SNI, Sham) separately, we use the **female time-point sequence** as the "control" trajectory and **male@day3** as `cf_0`. `reconstruct_cf` then propagates that initial male cloud forward using female dynamics, producing a predicted male trajectory at day14 and 5months.

At every time point we compare:
- **Predicted male** (WPT output) vs **Observed male**

using the two-sample energy statistic (Székely & Rizzo 2013) with a permutation p-value.

The prediction at day3 is trivially exact (it equals `cf_0`); the genuine predictions are at day14 and 5months.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import ot as pot
from src.pt import *

from src.cf_recon import reconstruct_cf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

out_dir = Path('../../outputs/GSE162807_mf')
out_dir.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

print('Libraries loaded.')
def w2_dist(A, B, max_pts=np.inf, seed=0):
    """W2 distance between two EmpiricalMeasures (or plain numpy arrays → uniform weights).
    Subsamples if clouds exceed max_pts."""
    rng = np.random.default_rng(seed)
    if isinstance(A, EmpiricalMeasure):
        xa, wa = np.asarray(A.locs, float), np.asarray(A.weights, float)
    else:
        xa = np.asarray(A, float); wa = np.ones(len(xa)) / len(xa)
    if isinstance(B, EmpiricalMeasure):
        xb, wb = np.asarray(B.locs, float), np.asarray(B.weights, float)
    else:
        xb = np.asarray(B, float); wb = np.ones(len(xb)) / len(xb)
    wa /= wa.sum(); wb /= wb.sum()
    if len(xa) > max_pts:
        idx = rng.choice(len(xa), max_pts, replace=False, p=wa)
        xa, wa = xa[idx], wa[idx]; wa /= wa.sum()
    if len(xb) > max_pts:
        idx = rng.choice(len(xb), max_pts, replace=False, p=wb)
        xb, wb = xb[idx], wb[idx]; wb /= wb.sum()
    M = pot.dist(xa, xb, metric='euclidean')
    return np.sqrt(pot.emd2(wa, wb, M, numItermax=1e10))

In [ ]:
# -- Load saved AnnData (from wpt_tansley_analysis.ipynb) ---------------------
H5AD_PATH = Path('../../outputs/GSE162807/tansley_preprocessed.h5ad')

if H5AD_PATH.exists():
    adata_full = sc.read_h5ad(H5AD_PATH)
    print(f'Loaded full dataset: {adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes')
else:
    raise FileNotFoundError(
        f'{H5AD_PATH} not found.\n'
        'Run wpt_tansley_analysis.ipynb first and save with:\n'
        '    adata.write_h5ad("../../outputs/GSE162807/tansley_preprocessed.h5ad")'
    )

print(adata_full.obs.groupby(['sex', 'condition', 'timepoint'], observed=True).size().to_string())

In [ ]:
# -- Optional stratified subsampling for faster evaluation ----------------------
SUBSAMPLE = False
N_PER_GROUP = 3000  # max cells per (sex, condition, timepoint) group

if SUBSAMPLE:
    rng = np.random.default_rng(42)
    keep_idx = []
    grouped = adata_full.obs.groupby(['sex', 'condition', 'timepoint'], observed=True).indices
    for _, idx in grouped.items():
        idx = np.asarray(idx)
        if len(idx) > N_PER_GROUP:
            idx = rng.choice(idx, N_PER_GROUP, replace=False)
        keep_idx.append(idx)
    keep_idx = np.concatenate(keep_idx) if keep_idx else np.array([], dtype=int)
    adata = adata_full[keep_idx].copy()
    print(f'Using subsampled adata: {adata.n_obs:,} cells (<= {N_PER_GROUP} per group)')
else:
    adata = adata_full.copy()
    print('Using full adata (no subsampling).')

print(adata.obs.groupby(['sex', 'condition', 'timepoint'], observed=True).size().to_string())

In [ ]:
# ── Standardised PCA (same convention as wpt_tansley_analysis) ────────────────
n_pcs   = 15
pca_raw = adata.obsm['X_pca'][:, :n_pcs]
pca_std = pca_raw.std(axis=0)
pca     = pca_raw / pca_std
obs     = adata.obs

TP_ORDER = ['day3', 'day14', '5months']

COL_FEMALE    = '#D62828'   # red   – female observed
COL_MALE_OBS  = '#2E86AB'   # blue  – male observed
COL_PREDICTED = '#2DC653'   # green – male predicted via WPT

def get_pca_mf(sex, cond, tp):
    mask = ((obs['sex'] == sex) &
            (obs['condition'] == cond) &
            (obs['timepoint'] == tp))
    return pca[mask.values]

print('Point cloud sizes:')
for sex in ['Male', 'Female']:
    for cond in ['SNI', 'Sham']:
        for tp in TP_ORDER:
            n = len(get_pca_mf(sex, cond, tp))
            if n > 0:
                print(f'  {sex:6s}  {cond:4s}  {tp:8s}: {n}')

### Run WPT

For each condition:
1. `control = [female@day3, female@day14, female@5months]`  — female trajectory drives the transport
2. `cf_0    = male@day3`  — initial male distribution (first shared timepoint)
3. `reconstruct_cf(control, cf_0)` → predicted male curve at day3, day14, 5months

`pred_curve[0]` == `cf_0` (day3, trivially exact); `pred_curve[1]` and `pred_curve[2]` are the genuine predictions at day14 and 5months.

In [ ]:
# ── Run WPT for SNI and Sham ──────────────────────────────────────────────────
results = {}   # cond -> {tps, female_traj, male_obs, pred_curve}

for cond in ['SNI', 'Sham']:
    female_traj  = [get_pca_mf('Female', cond, tp) for tp in TP_ORDER]
    male_obs     = [get_pca_mf('Male',   cond, tp) for tp in TP_ORDER]

    # keep only time points where both sexes have cells
    common_tps = [tp for tp, f, m in zip(TP_ORDER, female_traj, male_obs)
                  if len(f) > 0 and len(m) > 0]
    female_traj = [get_pca_mf('Female', cond, tp) for tp in common_tps]
    male_obs    = [get_pca_mf('Male',   cond, tp) for tp in common_tps]

    print(f'{cond}  ({len(common_tps)} time points: {common_tps})')
    for tp, f, m in zip(common_tps, female_traj, male_obs):
        print(f'  {tp:8s}: Female={len(f)}, Male={len(m)}')

    cf0     = male_obs[0]
    n_steps = len(common_tps) - 1

    print(f'  cf_0 = Male @ {common_tps[0]}: {len(cf0)} cells')
    print(f'  Running reconstruct_cf ({n_steps} steps, {n_pcs} PCs) ...')
    pred_curve = reconstruct_cf(female_traj, cf0, n=n_steps,
                                project=False, tol=1e-6)
    print(f'  Done.')

    male_obs_em = [EmpiricalMeasure(m, np.ones(len(m)) / len(m)) for m in male_obs]

    # pred_curve[0] is cf_0 passed through np.unique (sorted/deduplicated) by
    # wasserstein_expmap_vel, so it differs from male_obs[0] in ordering even
    # though they represent the same measure. Pin it back to the original.
    pred_curve[0] = male_obs_em[0]

    results[cond] = {
        'tps':         common_tps,
        'female_traj': [EmpiricalMeasure(f, np.ones(len(f)) / len(f)) for f in female_traj],
        'male_obs':    male_obs_em,
        'pred_curve':  pred_curve,
    }

In [ ]:
# ── Save results cache (run this once after run-wpt; enables rebuild-results) ─
import pickle, gzip

_cache_path = out_dir / 'mf_results_cache.pkl.gz'
_cache = {}
for cond, res in results.items():
    _cache[cond] = {
        'tps':         res['tps'],
        'female_traj': [(em.locs, em.weights) for em in res['female_traj']],
        'male_obs':    [(em.locs, em.weights) for em in res['male_obs']],
        'pred_curve':  [(em.locs, em.weights) for em in res['pred_curve']],
    }
with gzip.open(_cache_path, 'wb') as f:
    pickle.dump(_cache, f)
print(f'Saved → {_cache_path}  ({_cache_path.stat().st_size / 1e6:.1f} MB)')

### Trajectory visualisation (PC1–PC2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (cond, res) in zip(axes, results.items()):
    tps     = res['tps']
    f_traj  = res['female_traj']
    m_traj  = res['male_obs']
    p_curve = res['pred_curve']

    # weighted mean of an EmpiricalMeasure
    def wmean(em): return (em.weights[:, None] * em.locs).sum(0)

    f_means = np.array([wmean(s) for s in f_traj])
    m_means = np.array([wmean(s) for s in m_traj])
    p_means = np.array([wmean(p) for p in p_curve])

    ax.plot(f_means[:, 0], f_means[:, 1], 'o-',
            color=COL_FEMALE,    lw=2.5, ms=8, label='Female (observed)')
    ax.plot(m_means[:, 0], m_means[:, 1], 's-',
            color=COL_MALE_OBS,  lw=2.5, ms=8, label='Male (observed)')
    ax.plot(p_means[:, 0], p_means[:, 1], '^--',
            color=COL_PREDICTED, lw=2.5, ms=8, label='Male (predicted via WPT)')

    for i, tp in enumerate(tps):
        ax.annotate(tp, f_means[i, :2], textcoords='offset points',
                    xytext=(5, 5), fontsize=8, color='#555555')

    ax.set_xlabel('PC 1 (standardized)', fontsize=12)
    ax.set_ylabel('PC 2 (standardized)', fontsize=12)
    ax.set_title(f'{cond} — female-driven WPT vs observed male',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'mf_wpt_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

### Point-cloud scatter at each time point

In [ ]:
# ── One row per condition, one column per time point ──────────────────────────
for cond, res in results.items():
    tps     = res['tps']
    f_traj  = res['female_traj']
    m_traj  = res['male_obs']
    p_curve = res['pred_curve']

    fig, axes = plt.subplots(1, len(tps), figsize=(7 * len(tps), 6),
                              sharex=False, sharey=False)

    for ax, tp, f, m, p in zip(axes, tps, f_traj, m_traj, p_curve):
        ax.scatter(f.locs[:, 0], f.locs[:, 1], c=COL_FEMALE,    s=4, alpha=0.4,
                   label=f'Female obs  (n={len(f.locs)})')
        ax.scatter(m.locs[:, 0], m.locs[:, 1], c=COL_MALE_OBS,  s=4, alpha=0.4,
                   label=f'Male obs    (n={len(m.locs)})')
        ax.scatter(p.locs[:, 0], p.locs[:, 1], c=COL_PREDICTED, s=4, alpha=0.4,
                   label=f'Predicted   (n={len(p.locs)})')
        # weighted centroids
        for em, col in [(f, COL_FEMALE), (m, COL_MALE_OBS), (p, COL_PREDICTED)]:
            cx = (em.weights[:, None] * em.locs).sum(0)
            ax.scatter(cx[0], cx[1], c=col, s=200, marker='*',
                       edgecolors='k', linewidths=0.8, zorder=5)
        ax.set_xlabel('PC 1', fontsize=11)
        ax.set_ylabel('PC 2', fontsize=11)
        ax.set_title(f'{cond}  t={tp}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=8, markerscale=3)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_dir / f'mf_scatter_{cond.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

### W2 distance over time

Tracks three pairwise W2 distances at every time step:
- **Predicted vs Male (obs)** — how well WPT predicts male dynamics
- **Female vs Male (obs)** — baseline sex gap
- **Female vs Predicted** — how closely the prediction tracks the female trajectory

In [ ]:
# ── Rebuild `results` from raw PCA arrays (no WPT needed) ────────────────────
# Run this cell instead of run-wpt when you want to jump straight to the
# visualisation cells.  It rebuilds results (female + male; pred_curve=None)
# in a few seconds from the already-loaded pca array.
#
# If run-wpt already ran, results exists and this cell skips the rebuild.

if 'results' not in dir():
    print('Rebuilding results from PCA arrays (no WPT) ...')
    results = {}
    for cond in ['SNI', 'Sham']:
        female_traj = [get_pca_mf('Female', cond, tp) for tp in TP_ORDER]
        male_obs    = [get_pca_mf('Male',   cond, tp) for tp in TP_ORDER]
        common_tps  = [tp for tp, f, m in zip(TP_ORDER, female_traj, male_obs)
                       if len(f) > 0 and len(m) > 0]
        female_traj = [get_pca_mf('Female', cond, tp) for tp in common_tps]
        male_obs    = [get_pca_mf('Male',   cond, tp) for tp in common_tps]
        results[cond] = {
            'tps':         common_tps,
            'female_traj': [EmpiricalMeasure(f, np.ones(len(f)) / len(f)) for f in female_traj],
            'male_obs':    [EmpiricalMeasure(m, np.ones(len(m)) / len(m)) for m in male_obs],
            'pred_curve':  None,   # not available without running WPT
        }
    print('Done. pred_curve=None: predicted-male pane will be blank in UMAP plots.')
else:
    print('`results` already defined, skipping rebuild.')

# ── Shared UMAP constants (also defined here so cells below work standalone) ──
CATEGORIES = ['female', 'predicted male', 'male']
CAT_COLORS = {'female': COL_FEMALE, 'predicted male': COL_PREDICTED, 'male': COL_MALE_OBS}
TP_COLORS  = {'day3': '#F4A261', 'day14': '#E76F51', '5months': '#264653'}
GREY       = '#cccccc'

In [ ]:
# ── Per-condition UMAP: fit embedding, then plot group-coloured 3-pane ─────────
import umap as umap_lib

cond_embeddings = {}   # store for reuse in grid cell

for cond, res in results.items():
    tps = res['tps']

    locs_list, cat_labels, tp_labels = [], [], []
    for i, tp in enumerate(tps):
        pairs = [(res['female_traj'][i], 'female'),
                 (res['male_obs'][i],    'male')]
        if res['pred_curve'] is not None:
            pairs.append((res['pred_curve'][i], 'predicted male'))
        for em, cat in pairs:
            locs_list.append(em.locs)
            cat_labels.extend([cat] * len(em.locs))
            tp_labels.extend([tp]  * len(em.locs))

    X       = np.vstack(locs_list)
    cats    = np.array(cat_labels)
    tps_arr = np.array(tp_labels)

    print(f'Fitting UMAP for {cond} ({len(X):,} points) ...')
    emb = umap_lib.UMAP(n_components=2, random_state=42,
                        n_neighbors=30, min_dist=0.3).fit_transform(X)
    print('Done.')

    cond_embeddings[cond] = dict(emb=emb, cats=cats, tps_arr=tps_arr, tps=tps)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, cat in zip(axes, CATEGORIES):
        mask_hi = cats == cat
        ax.scatter(emb[~mask_hi, 0], emb[~mask_hi, 1],
                   c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
        if mask_hi.sum() > 0:
            ax.scatter(emb[mask_hi, 0],  emb[mask_hi, 1],
                       c=CAT_COLORS[cat], s=4, alpha=0.6, linewidths=0, rasterized=True)
        ax.set_title(cat.capitalize(), fontsize=13, fontweight='bold',
                     color=CAT_COLORS[cat])
        ax.set_xlabel('UMAP 1', fontsize=11); ax.set_ylabel('UMAP 2', fontsize=11)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    plt.suptitle(f'{cond} — UMAP by group', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(out_dir / f'mf_umap_{cond.lower()}_groups.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Per-condition UMAP: 3×3 grid (rows=group, cols=timepoint) ─────────────────
for cond, d in cond_embeddings.items():
    emb     = d['emb']
    cats    = d['cats']
    tps_arr = d['tps_arr']
    tps     = d['tps']

    fig, axes = plt.subplots(3, 3, figsize=(18, 18))

    for row, cat in enumerate(CATEGORIES):
        for col, tp in enumerate(TP_ORDER):
            ax = axes[row, col]
            mask = (cats == cat) & (tps_arr == tp)

            # everything else grey
            ax.scatter(emb[~mask, 0], emb[~mask, 1],
                       c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
            # only this group × timepoint coloured
            if mask.sum() > 0:
                ax.scatter(emb[mask, 0], emb[mask, 1],
                           c=TP_COLORS[tp], s=4, alpha=0.7, linewidths=0, rasterized=True)

            if row == 0:
                ax.set_title(tp, fontsize=12, fontweight='bold', color=TP_COLORS[tp])
            if col == 0:
                ax.set_ylabel(cat.capitalize(), fontsize=12, fontweight='bold',
                              color=CAT_COLORS[cat])
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    plt.suptitle(f'{cond} — UMAP by group × timepoint', fontsize=15,
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(out_dir / f'mf_umap_{cond.lower()}_grid.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Combined analysis (all treatments pooled)

Instead of stratifying by condition (SNI / Sham), we pool **all female cells** across treatments to build the reference trajectory and predict **all male cells** at each timepoint.

This tests whether the female trajectory can capture male dynamics at the population level, regardless of treatment.

In [ ]:
# ── Build pooled point clouds (all conditions) ────────────────────────────────
# Use adata.obs directly and build a positional integer index aligned with pca
_obs = adata.obs.reset_index(drop=True)
_pca = adata.obsm['X_pca'][:, :n_pcs]
_pca = _pca / _pca.std(axis=0)

def get_pca_pooled(sex, tp):
    mask = (_obs['sex'] == sex) & (_obs['timepoint'] == tp)
    return _pca[mask.values]

pooled_female = [get_pca_pooled('Female', tp) for tp in TP_ORDER]
pooled_male   = [get_pca_pooled('Male',   tp) for tp in TP_ORDER]

print('Pooled point cloud sizes:')
for tp, f, m in zip(TP_ORDER, pooled_female, pooled_male):
    print(f'  {tp:8s}: Female={len(f)}, Male={len(m)}')

In [ ]:
# ── Run WPT on pooled data ─────────────────────────────────────────────────────
cf0_pooled = pooled_male[0]
n_steps    = len(TP_ORDER) - 1

print(f'cf_0 = Male @ {TP_ORDER[0]}: {len(cf0_pooled)} cells')
print(f'Running reconstruct_cf ({n_steps} steps, {n_pcs} PCs) ...')
pred_curve_pooled = reconstruct_cf(pooled_female, cf0_pooled, n=n_steps,
                                   project=False, tol=1e-6)
print('Done.')

# Keep full EmpiricalMeasure objects; wrap raw arrays for consistent w2_dist calls
pred_pooled   = pred_curve_pooled   # list of EmpiricalMeasure
female_pooled = [EmpiricalMeasure(f, np.ones(len(f)) / len(f)) for f in pooled_female]
male_pooled   = [EmpiricalMeasure(m, np.ones(len(m)) / len(m)) for m in pooled_male]

In [ ]:
# ── Trajectory plot (pooled) ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

def wmean(em): return (em.weights[:, None] * em.locs).sum(0)

f_means = np.array([wmean(f) for f in female_pooled])
m_means = np.array([wmean(m) for m in male_pooled])
p_means = np.array([wmean(p) for p in pred_pooled])

ax.plot(f_means[:, 0], f_means[:, 1], 'o-',
        color=COL_FEMALE,    lw=2.5, ms=8, label='Female (observed)')
ax.plot(m_means[:, 0], m_means[:, 1], 's-',
        color=COL_MALE_OBS,  lw=2.5, ms=8, label='Male (observed)')
ax.plot(p_means[:, 0], p_means[:, 1], '^--',
        color=COL_PREDICTED, lw=2.5, ms=8, label='Male (predicted via WPT)')

for i, tp in enumerate(TP_ORDER):
    ax.annotate(tp, f_means[i, :2], textcoords='offset points',
                xytext=(5, 5), fontsize=9, color='#555555')

ax.set_xlabel('PC 1 (standardized)', fontsize=12)
ax.set_ylabel('PC 2 (standardized)', fontsize=12)
ax.set_title('All treatments pooled — female-driven WPT vs observed male',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'mf_wpt_trajectories_pooled.png', dpi=150, bbox_inches='tight')
plt.show()

### n_pcs sweep table (SNI, Sham, pooled)

Computes W2 error to observed male for two predictors at each non-initial timepoint:
- WPT prediction
- Mean-shift baseline

Runs this for three experiment modes: `SNI`, `Sham`, and pooled (all conditions together), then exports a LaTeX table.

In [ ]:
# -- n_pcs sweep table: SNI, Sham, and pooled experiments ----------------------
NPCS_GRID = [2, 3, 5, 7, 10, 15, 20]
MAX_W2_PTS = 5000
SEED = 42
EVAL_TPS = ['day14', '5months']

EXPERIMENTS = [
    ('Sham', 'Sham'),
    ('SNI', 'SNI'),
    ('Pooled', None),
]

def _uniform_em(X):
    X = np.asarray(X, float)
    return EmpiricalMeasure(X, np.ones(len(X)) / len(X))

def _weighted_mean(em):
    return (em.weights[:, None] * em.locs).sum(0)

def _eval_one_experiment(female_arrays, male_arrays, tps):
    # Keep only shared timepoints with non-empty clouds.
    keep = [i for i, (f, m) in enumerate(zip(female_arrays, male_arrays)) if len(f) > 0 and len(m) > 0]
    if len(keep) < 2:
        return {}

    f_arr = [female_arrays[i] for i in keep]
    m_arr = [male_arrays[i] for i in keep]
    tps_k = [tps[i] for i in keep]

    pred_curve = reconstruct_cf(f_arr, m_arr[0], n=len(tps_k) - 1, project=False, tol=1e-6)

    female_em = [_uniform_em(x) for x in f_arr]
    male_em = [_uniform_em(x) for x in m_arr]
    pred_curve[0] = male_em[0]

    disp = _weighted_mean(male_em[0]) - _weighted_mean(female_em[0])

    out = {}
    for i in range(1, len(tps_k)):
        tp = tps_k[i]
        naive = EmpiricalMeasure(np.asarray(female_em[i].locs) + disp, female_em[i].weights)
        out[tp] = {
            'WPT': w2_dist(pred_curve[i], male_em[i], max_pts=MAX_W2_PTS, seed=SEED),
            'MeanShift': w2_dist(naive, male_em[i], max_pts=MAX_W2_PTS, seed=SEED),
        }
    return out

# sweep_results[exp_name][n_pcs] = {tp: {'WPT':..., 'MeanShift':...}}
sweep_results = {name: {} for name, _ in EXPERIMENTS}

for n_pcs_eval in NPCS_GRID:
    print(f'Evaluating n_pcs={n_pcs_eval} ...')
    pca_eval = adata.obsm['X_pca'][:, :n_pcs_eval]
    pca_std = pca_eval.std(axis=0)
    pca_std[pca_std == 0] = 1.0
    pca_eval = pca_eval / pca_std
    obs_eval = adata.obs

    def _get_arr(sex, tp, cond=None):
        mask = (obs_eval['sex'] == sex) & (obs_eval['timepoint'] == tp)
        if cond is not None:
            mask = mask & (obs_eval['condition'] == cond)
        return pca_eval[mask.values]

    for exp_name, cond in EXPERIMENTS:
        female = [_get_arr('Female', tp, cond=cond) for tp in TP_ORDER]
        male = [_get_arr('Male', tp, cond=cond) for tp in TP_ORDER]
        sweep_results[exp_name][n_pcs_eval] = _eval_one_experiment(female, male, TP_ORDER)

    print('  done')

def _build_manual_latex_table(exp_name, exp_results, n_pcs_values, eval_tps):
    all_tps = [tp for tp in eval_tps if all(tp in exp_results[n] for n in n_pcs_values)]
    if not all_tps:
        raise ValueError(f'No shared non-initial timepoints found for {exp_name}.')

    lines = []
    col_spec = 'c' + 'cc' * len(all_tps)
    lines.append(r'\begin{table}[h]')
    lines.append(r'\centering')
    lines.append(r'\small')
    lines.append(r'\begin{tabular}{' + col_spec + '}')
    lines.append(r'\toprule')

    hdr1 = [rf'\multicolumn{{2}}{{c}}{{{tp}}}' for tp in all_tps]
    lines.append(r'$n_{\mathrm{pcs}}$ & ' + ' & '.join(hdr1) + r' \\')

    cmr = ''.join([rf'\cmidrule(lr){{{2 + 2*i}-{3 + 2*i}}}' for i in range(len(all_tps))])
    lines.append(cmr)

    hdr2 = []
    for _ in all_tps:
        hdr2.extend([r'WPT', r'Mean-shift'])
    lines.append(' & ' + ' & '.join(hdr2) + r' \\')
    lines.append(r'\midrule')

    for n in n_pcs_values:
        row = [str(n)]
        for tp in all_tps:
            row.append(f"{exp_results[n][tp]['WPT']:.3f}")
            row.append(f"{exp_results[n][tp]['MeanShift']:.3f}")
        lines.append(' & '.join(row) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(
        rf'\caption{{$W_2$ to observed male per non-initial timepoint for WPT and mean-shift baseline across PCA dimensions ({exp_name}).}}'
    )
    lines.append(rf'\label{{tab:mf_w2_npcs_{exp_name.lower()}}}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)

out_paths = {
    'Sham': out_dir / 'mf_w2_npcs_table_sham.tex',
    'SNI': out_dir / 'mf_w2_npcs_table_sni.tex',
    'Pooled': out_dir / 'mf_w2_npcs_table_pooled.tex',
}

for exp_name, _ in EXPERIMENTS:
    latex_table = _build_manual_latex_table(
        exp_name=exp_name,
        exp_results=sweep_results[exp_name],
        n_pcs_values=NPCS_GRID,
        eval_tps=EVAL_TPS,
    )
    out_paths[exp_name].write_text(latex_table)
    print('\n' + '=' * 60)
    print(f'{exp_name} table')
    print(latex_table)
    print(f'Saved -> {out_paths[exp_name]}')

In [ ]:
import umap

# -- Use results from memory; never overwrite the global ------------------------
assert 'results' in dir(), (
    "Run rebuild-results (or run-wpt) first to populate `results`."
 )
_res = results

# -- Collect all point clouds with labels ---------------------------------------
all_locs, all_labels, all_tps, all_conds = [], [], [], []

for cond, res in _res.items():
    for i, tp in enumerate(res['tps']):
        f = res['female_traj'][i]
        m = res['male_obs'][i]
        p = res['pred_curve'][i] if res['pred_curve'] is not None else None

        all_locs.append(f.locs)
        all_labels.extend(['female'] * len(f.locs))
        all_locs.append(m.locs)
        all_labels.extend(['male'] * len(m.locs))
        n_total = len(f.locs) + len(m.locs)

        # Avoid passing duplicate points to UMAP at the initial timepoint
        # because pred_curve[0] and male_obs[0] are the same measure.
        if p is not None and i > 0:
            all_locs.append(p.locs)
            all_labels.extend(['predicted male'] * len(p.locs))
            n_total += len(p.locs)

        all_tps.extend([tp] * n_total)
        all_conds.extend([cond] * n_total)

X_all = np.vstack(all_locs)
labels = np.array(all_labels)
tps_arr = np.array(all_tps)
cond_arr = np.array(all_conds)

print(f'Labels present: {np.unique(labels)}')
print(f'Fitting UMAP on {len(X_all):,} points x {X_all.shape[1]} dims ...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
embedding = reducer.fit_transform(X_all)
print('Done.')

df_umap = pd.DataFrame({
    'UMAP1': embedding[:, 0],
    'UMAP2': embedding[:, 1],
    'label': labels,
    'tp': tps_arr,
    'cond': cond_arr,
})

In [ ]:
# ── Pooled UMAP: 3-pane by group ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cat in zip(axes, CATEGORIES):
    mask_hi = labels == cat
    ax.scatter(embedding[~mask_hi, 0], embedding[~mask_hi, 1],
               c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
    if mask_hi.sum() > 0:
        ax.scatter(embedding[mask_hi, 0],  embedding[mask_hi, 1],
                   c=CAT_COLORS[cat], s=4, alpha=0.6, linewidths=0, rasterized=True)
    ax.set_title(cat.capitalize(), fontsize=13, fontweight='bold', color=CAT_COLORS[cat])
    ax.set_xlabel('UMAP 1', fontsize=11); ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

plt.suptitle('All treatments pooled — UMAP by group', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'mf_umap_pooled_groups.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- Pooled UMAP: 3 x N grid (rows=group, cols=timepoint) ----------------------
n_tp = len(TP_ORDER)
n_cats = len(CATEGORIES)
fig, axes = plt.subplots(n_cats, n_tp, figsize=(6 * n_tp, 6 * n_cats))

if n_cats == 1:
    axes = axes[np.newaxis, :]
if n_tp == 1:
    axes = axes[:, np.newaxis]

# Match chimp notebook: avoid yellow for the final timepoint.
if TP_ORDER:
    TP_COLORS[TP_ORDER[-1]] = '#2A9D8F'  # teal

for row, cat in enumerate(CATEGORIES):
    for col, tp in enumerate(TP_ORDER):
        ax = axes[row, col]
        mask = (labels == cat) & (tps_arr == tp)

        # Match chimp behavior: predicted initial panel should still show points
        # even though those duplicate points were excluded from UMAP input.
        if cat == 'predicted male' and tp == TP_ORDER[0]:
            mask |= (labels == 'male') & (tps_arr == TP_ORDER[0])

        ax.scatter(embedding[~mask, 0], embedding[~mask, 1],
                   c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
        if mask.sum() > 0:
            tp_col = TP_COLORS[tp]
            ax.scatter(embedding[mask, 0], embedding[mask, 1],
                       c=[tp_col], s=4, alpha=0.7, linewidths=0, rasterized=True)

        if row == 0:
            ax.set_title(tp, fontsize=12, fontweight='bold', color=TP_COLORS[tp])
        if col == 0:
            ax.set_ylabel(cat.capitalize(), fontsize=12, fontweight='bold',
                          color=CAT_COLORS[cat])
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

plt.suptitle('All treatments pooled — UMAP by group x timepoint',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(out_dir / 'mf_umap_pooled_grid.png', dpi=150, bbox_inches='tight')
plt.show()